In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def clean_descriptive(df:pd.DataFrame, verbose:bool=False)-> pd.DataFrame:
    df = df.copy() # defensive copy
    
    df['DONOR_GENDER'] = df['DONOR_GENDER'].fillna('U').str.upper()
    
    df['HOME_OWNER'] = df['HOME_OWNER'].fillna('U').str.upper()
    df['HOME_OWNER'] = df['HOME_OWNER'].map({'H': 1, 'U': 0}).astype(int)
    
    df['RECENCY_STATUS_96NK'] = df['RECENCY_STATUS_96NK'].fillna('U').str.upper()
    
    df['SES'] = df['SES'].replace('?', np.nan).astype(float)
    df['INC_BUCKET'] = pd.qcut(df['MEDIAN_HOUSEHOLD_INCOME'], q=4, labels=False, duplicates='drop')
    # 3. Create a clean subset of rows that contain real, un-imputed SES data
    clean_subset = df[df['SES'].notna()]
    verified_ses_medians = clean_subset.groupby('INC_BUCKET')['SES'].median()
    df['SES'] = df['SES'].fillna(df['INC_BUCKET'].map(verified_ses_medians))
    df['SES'] = df['SES'].fillna(df['SES'].median()).round().astype(int) # Fallback
    df.drop(columns=['INC_BUCKET'], inplace=True)
    
    df['URBANICITY'] = df['URBANICITY'].replace('?', None).fillna('Unknown').str.upper()
    
    df['PEP_STAR'] = df['PEP_STAR'].apply(lambda x: np.nan if x < 0 else (1.0 if x > 1 else x))
    df['PEP_STAR_IS_MISSING'] = np.where(df['PEP_STAR'].isna(), 1, 0) # preserve missingness as a feature, since it may be informative
    df['PEP_STAR'] = np.where(df['PEP_STAR'] > 0.5, 1, 0).astype(int)
    # 2. Bucket lifetime gift counts into 3 tiers (Low, Medium, High frequency)
    df['GIFT_COUNT_TIER'] = pd.qcut(df['LIFETIME_GIFT_COUNT'], q=3, labels=False, duplicates='drop')
    group_modes = df.groupby('GIFT_COUNT_TIER')['PEP_STAR'].transform(lambda x: x.mode()[0] if not x.mode().empty else 0)
    df['PEP_STAR'] = df['PEP_STAR'].fillna(group_modes).round().astype(int)
    df.drop(columns=['GIFT_COUNT_TIER'], inplace=True)
    
    df['FREQUENCY_STATUS_97NK'] = df['FREQUENCY_STATUS_97NK'].apply(lambda x: np.nan if x < 1 else (4.0 if x > 4 else x))
    df['FREQUENCY_STATUS_97NK_IS_MISSING'] = np.where(df['FREQUENCY_STATUS_97NK'].isna(), 1, 0)
    df['FREQUENCY_STATUS_97NK'] = df['FREQUENCY_STATUS_97NK'].fillna(-1).round().astype(int)
    
    df['CHILDREN'] = df['CHILDREN'].apply(lambda x: np.nan if x < 0 else (4.0 if x > 4 else x))
    df['CHILDREN_IS_MISSING'] = np.where(df['CHILDREN'].isna(), 1, 0)
    df['CHILDREN'] = df['CHILDREN'].fillna(0).round().astype(int)
    
    df['INCOME_GROUP'] = np.where(df['INCOME_GROUP'] > 7, 7, df['INCOME_GROUP'])
    df['INCOME_GROUP'] = np.where((df['INCOME_GROUP'] < 1) | (df['INCOME_GROUP'].isna()), -1, df['INCOME_GROUP'])
    df['INCOME_GROUP'] = df['INCOME_GROUP'].round().astype(int)
    
    df['WEALTH_RATING'] = np.where(df['WEALTH_RATING'] > 9, 9.0, df['WEALTH_RATING'])
    df['WEALTH_RATING'] = np.where((df['WEALTH_RATING'] < 0) | (df['WEALTH_RATING'].isna()), -1.0, df['WEALTH_RATING'])
    df['WEALTH_RATING'] = df['WEALTH_RATING'].round().astype(int)
    
    df['RECENT_CARD_RESPONSE_COUNT_IS_MISSING'] = np.where(df['RECENT_CARD_RESPONSE_COUNT'].isna(), 1, 0)
    df['RECENT_CARD_RESPONSE_COUNT'] = np.where(
        df['RECENT_CARD_RESPONSE_COUNT'] > 0, 
        df['RECENT_CARD_RESPONSE_COUNT'].round(), 
        df['RECENT_CARD_RESPONSE_COUNT']
    )
    df['RECENT_CARD_RESPONSE_COUNT'] = np.where(
        (df['RECENT_CARD_RESPONSE_COUNT'] < 0) | (df['RECENT_CARD_RESPONSE_COUNT'].isna()), 
        0, 
        df['RECENT_CARD_RESPONSE_COUNT']
    ).astype(int)
    
    df['RECENT_RESPONSE_COUNT_IS_MISSING'] = np.where(df['RECENT_RESPONSE_COUNT'].isna(), 1, 0)
    df['RECENT_RESPONSE_COUNT'] = np.where(
        df['RECENT_RESPONSE_COUNT'] > 0, 
        df['RECENT_RESPONSE_COUNT'].round(), 
        df['RECENT_RESPONSE_COUNT']
    )
    df['RECENT_RESPONSE_COUNT'] = np.where(
        (df['RECENT_RESPONSE_COUNT'] < 0) | (df['RECENT_RESPONSE_COUNT'].isna()), 
        0, 
        df['RECENT_RESPONSE_COUNT']
    ).astype(int)
    
    df['CARD_PROM_12_IS_MISSING'] = np.where(df['CARD_PROM_12'].isna(), 1, 0)
    df['CARD_PROM_12'] = np.where(df['CARD_PROM_12'].notna(), df['CARD_PROM_12'].round(), df['CARD_PROM_12'])
    df['CARD_PROM_12'] = df['CARD_PROM_12'].fillna(0).astype(int)
    
    df['RECENT_STAR_STATUS'] = np.where(df['RECENT_STAR_STATUS'].isna(), 1, 0)
    df['RECENT_STAR_STATUS'] = np.where(df['RECENT_STAR_STATUS'] < 0, 0, df['RECENT_STAR_STATUS'])
    df['RECENT_STAR_STATUS'] = np.where(df['RECENT_STAR_STATUS'] > 0, 1, df['RECENT_STAR_STATUS'])
    df['RECENT_STAR_STATUS'] = df['RECENT_STAR_STATUS'].astype(int)
    
    df['DONOR_AGE_IS_MISSING'] = np.where(df['DONOR_AGE'].isna(), 1, 0)
    # Segment continuous columns into 3 groups (Low, Med, High tiers)
    # This creates clean groups even if wealth rating is empty
    df['INCOME_TIER'] = pd.qcut(df['MEDIAN_HOUSEHOLD_INCOME'], q=3, labels=['Low_Inc', 'Med_Inc', 'High_Inc'])
    df['TENURE_TIER'] = pd.qcut(df['MONTHS_SINCE_FIRST_GIFT'], q=3, labels=['New_Donor', 'Mid_Donor', 'Loyal_Donor'])
    robust_group_cols = ['URBANICITY', 'HOME_OWNER', 'INCOME_TIER', 'TENURE_TIER']
    group_medians = df.groupby(robust_group_cols)['DONOR_AGE'].transform('median')
    df['DONOR_AGE'] = df['DONOR_AGE'].fillna(group_medians)
    overall_median = df['DONOR_AGE'].median() # fallback if any group is empty
    df['DONOR_AGE'] = df['DONOR_AGE'].fillna(overall_median)
    df['DONOR_AGE'] = df['DONOR_AGE'].round().astype(int)
    df.drop(columns=['INCOME_TIER', 'TENURE_TIER'], inplace=True)

    # Cleaning remaining columns
    discrete_cols = [
        'MONTHS_SINCE_LAST_GIFT', 'FILE_CARD_GIFT', 'MONTHS_SINCE_LAST_PROM_RESP',
        'NUMBER_PROM_12', 'LIFETIME_CARD_PROM', 'LIFETIME_GIFT_COUNT',
        'MONTHS_SINCE_FIRST_GIFT', 'LIFETIME_PROM'
    ]
    for col in discrete_cols:
        df[col] = np.round(pd.to_numeric(df[col], errors='coerce'))
        df[col] = df[col].clip(lower=0).fillna(df[col].median()).astype(int)

    proportions_n_percentages_cols = [
        'RECENT_CARD_RESPONSE_PROP', 'PCT_ATTRIBUTE1', 'PCT_ATTRIBUTE2', 
        'PCT_ATTRIBUTE3', 'PCT_ATTRIBUTE4', 'PCT_OWNER_OCCUPIED', 'RECENT_RESPONSE_PROP'
    ]
    for col in proportions_n_percentages_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Check if the column is a 0-1 proportion or a 0-100 percentage
        if df[col].max() <= 1.0:
            df[col] = df[col].clip(lower=0.0, upper=1.0)
        else:
            df[col] = df[col].clip(lower=0.0, upper=100.0)
        df[col] = df[col].fillna(df[col].median())

    monetary_cols = [
        'LIFETIME_MIN_GIFT_AMT', 'LAST_GIFT_AMT', 'LIFETIME_MAX_GIFT_AMT', 
        'RECENT_AVG_CARD_GIFT_AMT', 'RECENT_AVG_GIFT_AMT', 'LIFETIME_GIFT_AMOUNT', 
        'MEDIAN_HOUSEHOLD_INCOME', 'MEDIAN_HOME_VALUE', 'PER_CAPITA_INCOME'
    ]
    for col in monetary_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].clip(lower=0.01) #expecting donations can't be zero.
        df[col] = df[col].fillna(df[col].median())

    # print("--- SES from INCOME_GROUP ---")
    # df['SES'] = df.groupby('INCOME_GROUP')['SES'].transform(
    #     lambda x: x.fillna(x.mode()[0] if not x.mode().empty else df['SES'].median())
    # )

    df['URBANICITY'] = df['URBANICITY'].replace('?', np.nan)
    # group by home value quintiles to find the urbanicity mode
    home_value_groups = pd.qcut(df['MEDIAN_HOME_VALUE'], q=5, duplicates='drop')
    global_mode = df['URBANICITY'].mode()[0]
    df['URBANICITY'] = df.groupby(home_value_groups)['URBANICITY'].transform(
        lambda g: g.fillna(g.mode()[0] if not g.mode().empty else global_mode)
    )

    missing_pct = df.isnull().mean() * 100
    uncleaned = (missing_pct[missing_pct > 0].sort_values(ascending=False))
    assert uncleaned.empty, f"Data still has missing values after cleaning:\n{uncleaned}"

    return df